# **Ujjwal Karki: BERT Implementation on Cleaned Dataset**

## 1. Import Libraries

In [28]:
import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from tqdm.auto import tqdm
SEED = 42

# 2. Load and Prepare Dataset

## *2.1. Load Dataset*

In [29]:
df = pd.read_csv('../data/cleaned_news.csv')
df.head()

,target,title_clean,text_clean
0,0,smell hillary’s fear,"daniel greenfield, shillman journalism fellow ..."
1,0,watch exact moment paul ryan committed politic...,google pinterest digg linkedin reddit stumbleu...
2,1,kerry go paris gesture sympathy,u.s. secretary state john f. kerry said monday...
3,0,bernie supporters twitter erupt anger dnc: 'we...,"— kaydee king (@kaydeeking) november 9, 2016 l..."
4,1,battle new york: primary matters,primary day new york front-runners hillary cli...


## *2.2. Prepare Dataset* 

In [30]:
# remove rows with missing targets
df = df.dropna(subset=["target"]).copy()

# ensure text cols contain strings
df['title_clean'] = df['title_clean'].fillna("").astype(str)
df['text_clean'] = df['text_clean'].fillna("").astype(str)

# ensure labels are integers
df['target'] = df['target'].astype(int)

# keep only valid labels
df = df[df['target'].isin([0, 1])].reset_index(drop=True)

print(df.head())
print(df['target'].value_counts())

   target                                        title_clean  \
0       0                               smell hillary’s fear   
1       0  watch exact moment paul ryan committed politic...   
2       1                    kerry go paris gesture sympathy   
3       0  bernie supporters twitter erupt anger dnc: 'we...   
4       1                   battle new york: primary matters   

                                          text_clean  
0  daniel greenfield, shillman journalism fellow ...  
1  google pinterest digg linkedin reddit stumbleu...  
2  u.s. secretary state john f. kerry said monday...  
3  — kaydee king (@kaydeeking) november 9, 2016 l...  
4  primary day new york front-runners hillary cli...  
target
1    3149
0    3107
Name: count, dtype: int64


# 3. Implement BERT

## *3.1. Set Device CUDA*

In [31]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## *3.2. Split the Dataset*

In [32]:
train_df, test_df = train_test_split(
    df,
    test_size = 0.20,
    random_state=SEED,
    stratify=df['target']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.125,
    random_state=SEED,
    stratify=train_df['target']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(train_df["target"].value_counts(normalize=True))

Training samples: 4378
Validation samples: 626
Testing samples: 1252

Training distribution:
target
1    0.503426
0    0.496574
Name: proportion, dtype: float64


## *3.3. Load Tokenizer*

In [33]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

## *3.4. PyTorch Dataset*

In [34]:
class NewsDataset(Dataset):
    """
    BERT expects a consistent data format which traditional pandas dataframe can't deliver
    which is why a custom pytorch dataset class is used 
    Flow: Pandas DF -> PyTorch Dataset -> Data Loader -> Batches of Tensors -> BERT Model 
    """
    def __init__(self, dataframe, tokenizer, max_length):
        self.titles = dataframe['title_clean'].tolist()
        self.texts = dataframe['text_clean'].tolist()
        self.labels = dataframe['target'].tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        title = self.titles[index]
        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(
            title,
            text,
            add_special_tokens=True,
            max_length = self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            'token_type_ids': encoding['token_type_ids'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }    

# 4. Evaluate Raw Performance

# 5. Hyperparameter Tuning

# 6. Re-Evaluate Performance